# Milestone 4 demo: load → infer → detect drift → query human → update model

This notebook walks one cycle of the deployed uplift system:

1. **Load** the Milestone 2 X-learner (v1) and the data.
2. **Infer**: score users, pick the top 10% to target, and measure latency.
3. **Detect drift**: a new batch arrives with all raw features shifted by 3 sd; the monitor and the Qini trigger check it.
4. **Query human**: the reviewer approves the alert, and the model picks the users whose outcome is most uncertain; their outcomes are revealed by the (simulated) randomized experiment.
5. **Update model**: warm-start update on the queried labels plus the replay buffer, promotion gate, reviewer approval, and a saved model version.

Set `INTERACTIVE = True` to answer the reviewer's questions yourself. Full data takes about 6-10 minutes on Colab; set `FRAC = 0.2` for a quick run. Outputs are written to `outputs/demo/`.

In [ ]:
# SETUP: install the XGBoost version the saved model was written with, and make the pipeline importable.
# The package is looked for (1) next to this notebook, (2) in an uploaded Milestone_4_code.zip,
# (3) in a clone of the GitHub repository (pulled again if it already exists).
%pip install -q xgboost==3.4.1 psutil
import os, sys, glob, zipfile, subprocess
PKG = None
if os.path.isdir("uplift_pipeline"):
    PKG = "."
if PKG is None and glob.glob("Milestone_4_code*.zip"):
    zipfile.ZipFile(sorted(glob.glob("Milestone_4_code*.zip"))[-1]).extractall(".")
    PKG = "Milestone 4"
if PKG is None:
    if os.path.isdir("INSE-6450"):
        subprocess.run(["git", "-C", "INSE-6450", "pull", "-q"], check=False)
    else:
        subprocess.run(["git", "clone", "-q", "https://github.com/saopa28/INSE-6450.git"], check=True)
    PKG = os.path.join("INSE-6450", "Milestone 4")
if not os.path.isdir(os.path.join(PKG, "uplift_pipeline")):
    raise FileNotFoundError(f"uplift_pipeline not found in {PKG!r}. Push the 'Milestone 4' folder to GitHub, "
                            "or upload Milestone_4_code.zip to the Colab file panel and run this cell again.")
sys.path.insert(0, os.path.abspath(PKG))
import xgboost; print("xgboost", xgboost.__version__, "| pipeline from", os.path.abspath(PKG))

In [ ]:
import json, time
import numpy as np, pandas as pd
from IPython.display import display
from uplift_pipeline import load_data, split, shift, raw_sd, load_v1, summary, paired_boot, Monitor, qini_check, qini_floor, CONTINUAL, OUT_DIR, XLearner
from uplift_pipeline.continual import ReplayBuffer, latency
from uplift_pipeline.hitl import SimulatedOracle, SimulatedReviewer, select

FRAC, DATA, INTERACTIVE, SEED = 1.0, None, False, 42
OUT = OUT_DIR / "demo"; OUT.mkdir(parents=True, exist_ok=True)
rng = np.random.default_rng(SEED)

## 1. Load

In [ ]:
df = load_data(frac=FRAC) if DATA is None else load_data(DATA, frac=FRAC)
sp = split(df); del df
(Xtr, ttr, ytr), (Xva, tva, yva), (Xte, tte, yte) = sp["train"], sp["val"], sp["test"]
model = load_v1()
print("Loaded v1:", model.trees(), f"| scoring size {model.size_mb():.2f} MB")

## 2. Infer

In [ ]:
s = model.score(Xte)
top = np.argsort(-s)[: len(s) // 10]
print("Test metrics:", {k: round(v, 6) for k, v in summary(yte, tte, s).items()})
print(f"Target the top 10%: {len(top):,} users with uplift score >= {s[top[-1]]:.5f}")
p50, p90, _ = latency(model, Xte, 1); b50, b90, thr = latency(model, Xte, 256)
print(f"Latency: single user p50 {p50:.2f} ms, p90 {p90:.2f} ms | batch of 256 p50 {b50:.2f} ms ({thr:,.0f} users/s)")

## 3. Detect drift
A new day of traffic arrives in which all 12 raw features are shifted by +3 sd (the severe shift from Milestone 3). Labelled users from that day are split into the update pool and the gate rows used by the Qini check and the promotion gate.

In [ ]:
sd = raw_sd(Xtr)
idx = rng.permutation(len(ytr))[: min(1_000_000, len(ytr) // 4)]
X_new = shift(Xtr[idx], sd, 3.0); t_new, y_new = ttr[idx], ytr[idx]
cut = int(0.75 * len(idx))
pool = (X_new[:cut], t_new[:cut], y_new[:cut]); gate = (X_new[cut:], t_new[cut:], y_new[cut:])

monitor = Monitor(Xtr)
report = monitor.check(pool[0])
print("Monitor:", report["status"], report["alerts"], f"max PSI {report['psi_max']:.2f} ({report['psi_feature']})")
floor = qini_floor(yva, tva, model.score(Xva), len(gate[2]) // 2)[0]
triggered, windows = qini_check(model, *gate, floor=floor)
print(f"Qini trigger: {triggered} (window Qini {windows[0]:+.6f}, {windows[1]:+.6f}; floor {floor:.6f})")

## 4. Query human
The reviewer decides whether the alert opens an update. The model then asks for the outcomes of the users it is least sure about (largest disagreement between its two uplift estimators, |tau1 - tau0|), plus 30% random users for coverage. The oracle is the randomized experiment: it reveals treatment and conversion of the queried users.

In [ ]:
reviewer = SimulatedReviewer(interactive=INTERACTIVE)
approved = reviewer.review_alert(report, triggered)
print("Reviewer approved the update:", approved)

BUDGET = min(200_000, len(pool[2]) // 2)
available = np.ones(len(pool[2]), bool)
q = select(model, pool[0], available, BUDGET, "hybrid", rng)
oracle = SimulatedOracle(pool[1], pool[2])
t_q, y_q = oracle.query(q)
print(f"Queried {oracle.queried:,} users: treated share {t_q.mean():.3f}, conversion rate {y_q.mean():.4f}")

## 5. Update model
Warm-start update: the old trees of all four boosters are kept unchanged and at most 200 new trees (learning rate 0.05) are added per booster, trained on the queried users plus a replay sample of earlier (clean) data. The candidate is promoted only if the gate passes and the reviewer approves.

In [ ]:
buffer = ReplayBuffer(CONTINUAL["buffer_size"], Xtr.shape[1], SEED)
old = rng.choice(len(ytr), min(len(ytr), 2 * CONTINUAL["buffer_size"]), replace=False)
buffer.add(Xtr[old], ttr[old], ytr[old])             # earlier traffic seen by the deployed model
Xb, tb, yb = buffer.get()
X_up = np.concatenate([pool[0][q], Xb]); t_up = np.concatenate([t_q, tb]); y_up = np.concatenate([y_q, yb])
v = rng.choice(len(yva), min(len(yva), 300_000), replace=False)
val = (shift(Xva[v], sd, 3.0), tva[v], yva[v])        # early-stopping rows from the new distribution

t0 = time.perf_counter()
candidate = XLearner.fit(X_up, t_up, y_up, *val, init=model, new_trees=CONTINUAL["new_trees"],
                         lr=CONTINUAL["warm_lr"], early_stop=CONTINUAL["warm_early_stop"]) if approved else None
upd_s = time.perf_counter() - t0
if candidate is not None:
    ci, p = paired_boot(gate[2], gate[1], candidate.score(gate[0]), model.score(gate[0]), n=CONTINUAL["gate_boot"])
    print(f"Update took {upd_s:.1f} s | trees {model.trees()} -> {candidate.trees()}")
    promote = reviewer.review_promotion(ci, "v2-demo")
    print("Promoted:", promote)

In [ ]:
X_drift = shift(Xte, sd, 3.0)
rows = [dict(model="v1", **{f"{k} (drifted)": v for k, v in summary(yte, tte, model.score(X_drift)).items()},
             **{f"{k} (original)": v for k, v in summary(yte, tte, model.score(Xte)).items()})]
if candidate is not None:
    rows.append(dict(model="v2-demo", **{f"{k} (drifted)": v for k, v in summary(yte, tte, candidate.score(X_drift)).items()},
                     **{f"{k} (original)": v for k, v in summary(yte, tte, candidate.score(Xte)).items()}))
cmp = pd.DataFrame(rows).set_index("model"); display(cmp.round(6))

if candidate is not None and promote:
    man = candidate.save(OUT / "models" / "v2-demo", version="v2-demo", parent="v1", drift="all raw +3 sd",
                         labels_queried=int(oracle.queried), replay_rows=int(len(yb)), update_seconds=round(upd_s, 2),
                         gate_qini_gain_ci=list(ci))
    reloaded = XLearner.load(OUT / "models" / "v2-demo")
    assert np.allclose(reloaded.score(X_drift[:10000]), candidate.score(X_drift[:10000]))
    model = reloaded                                    # the new deployed model
    print("Saved and reloaded", man["version"], "| hashes", man["artifact_sha256"])

cmp.to_csv(OUT / "demo_comparison.csv")
pd.DataFrame(reviewer.log).to_csv(OUT / "review_log.csv", index=False)
(OUT / "demo_summary.json").write_text(json.dumps(dict(monitor=report["status"], psi_max=report["psi_max"],
    trigger=triggered, window_qini=windows, floor=floor, labels_queried=int(oracle.queried),
    update_seconds=upd_s, reviewer=reviewer.log), indent=2, default=float))
print("Outputs:", sorted(p.name for p in OUT.iterdir()))